# Stage 3 - Feature engineering

**Data: real Porto taxi trajectories** (UCI 339, CC BY 4.0) - a deterministic sample of trips,
one GPS ping every 15 s. **Not synthetic.** Everything below is computed from measured GPS.

**Limitation repeated everywhere results are reported:** taxis, not buses. No fixed stops, no timetable,
no schedule delay. Dwell here means "below walking pace", which on a taxi is kerbside waiting or traffic,
not boarding passengers.

Built here: H3 cell aggregates, trip-level features, cyclic time encoding, a 168-bucket weekly profile,
and standardisation. Outputs go to `data/processed/` (intermediates), not `outputs/`, which Stage 6-8 own.

In [1]:
import json, pathlib, sys, time
sys.path.append(str(pathlib.Path.cwd().parent))

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

from src.io import PROJECT_ROOT
from src.porto import load_porto_points, PORTO_ZIP
from src import cleaning as C
from src import features as F

PROC = PROJECT_ROOT / "data" / "processed"; PROC.mkdir(parents=True, exist_ok=True)
INT = PROJECT_ROOT / "data" / "interim"; INT.mkdir(parents=True, exist_ok=True)
FIG = PROJECT_ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 150)

In [2]:
# 1) Load and parse the sample (streams the CSV, parses polylines)
t0 = time.time()
points, meta = load_porto_points(PORTO_ZIP)
print(f"loaded {meta['points']:,} points from {meta['trips_sampled']:,} trips "
      f"in {time.time()-t0:.1f}s")
print(json.dumps({k: meta[k] for k in ['rows_scanned','rows_sampled_before_parsing',
                                       'trips_skipped_unparsable','sampling_interval_s',
                                       'timezone_local']}, indent=1))

loaded 2,927,093 points from 58,706 trips in 35.8s
{
 "rows_scanned": 1710670,
 "rows_sampled_before_parsing": 60000,
 "trips_skipped_unparsable": 1293,
 "sampling_interval_s": 15,
 "timezone_local": "Europe/Lisbon"
}


In [3]:
# 2) Clean with the Stage 2 pipeline. Time column = UTC (absolute): local wall
# clock would jump by an hour at a DST boundary and corrupt the time steps.
# The bounding box is derived from the data's 0.5-99.5 percentiles + margin: this is a
# COVERAGE DECISION (single-city analysis), not an error filter, and it is reported.
lat_q = points["lat"].quantile([0.005, 0.995])
lon_q = points["lon"].quantile([0.005, 0.995])
margin = 0.05
bbox = {"lat_min": float(lat_q.iloc[0] - margin), "lat_max": float(lat_q.iloc[1] + margin),
        "lon_min": float(lon_q.iloc[0] - margin), "lon_max": float(lon_q.iloc[1] + margin),
        "margin_deg": margin, "source": "0.5-99.5 percentile of sampled points + margin"}
print({k: round(v, 4) for k, v in bbox.items() if isinstance(v, float)})

clean, report = C.clean_points(points, bbox, group_col="trip_id", time_col="timestamp_utc")
print(f"points {report['rows_in']:,} -> {report['rows_out']:,} ({report['pct_removed']}% removed)")
for k in ["dropped_duplicates_total", "dropped_outside_bbox", "dropped_nonpositive_timestep",
          "dropped_impossible_jumps", "residual_implausible_steps"]:
    print(f"  {k:32s} {report[k]}")

{'lat_min': 40.9773, 'lat_max': 41.3857, 'lon_min': -8.7457, 'lon_max': -8.3344, 'margin_deg': 0.05}


points 2,927,093 -> 2,852,081 (2.563% removed)
  dropped_duplicates_total         0
  dropped_outside_bbox             29967
  dropped_nonpositive_timestep     0
  dropped_impossible_jumps         45045
  residual_implausible_steps       24246


In [4]:
# 3) Prepare: H3 cell (res 8), hour/weekday features, cyclic encoding, valid-step flag
t0 = time.time()
prepared = F.prepare_points(clean, group_col="trip_id")
print(f"prepared in {time.time()-t0:.1f}s")
print("step_valid share:", round(prepared['step_valid'].mean(), 4))
print("cells:", prepared['h3_cell'].nunique())
prepared[["trip_id","timestamp_local","lat","lon","speed_kmh","hour","weekday","h3_cell"]].head(3)

prepared in 6.3s
step_valid share: 0.9566
cells: 976


,trip_id,timestamp_local,lat,lon,speed_kmh,hour,weekday,h3_cell
0,1372636858620000589,2013-07-01 01:00:58+01:00,41.141412,-8.618643,NaN,1,0,8839220f07fffff
1,1372636858620000589,2013-07-01 01:01:13+01:00,41.141376,-8.618499,3.049342,1,0,8839220f07fffff
2,1372636858620000589,2013-07-01 01:01:28+01:00,41.142510,-8.620326,47.581961,1,0,8839220f07fffff


In [5]:
# 4) Cell-level features (one row per hexagon)
cells = F.cell_features(prepared).reset_index()
MIN_VISITS = 5   # a cell seen by <5 trips is one vehicle's noise, not a pattern
kept = cells[cells["n_visits"] >= MIN_VISITS].copy()
print(f"cells: {len(cells)}  kept (>= {MIN_VISITS} visits): {len(kept)}")
kept[["n_visits","n_points","dwell_s_per_visit","speed_mean","speed_p95","speed_cv",
      "slow_share","night_share","am_peak_share","peak_hour"]].describe().round(3)

cells: 976  kept (>= 5 visits): 602


,n_visits,n_points,dwell_s_per_visit,speed_mean,speed_p95,speed_cv,slow_share,night_share,am_peak_share,peak_hour
count,602.000,602.000,602.000,602.000,602.000,602.000,602.000,602.000,602.000,602.000
mean,661.254,4730.512,47.010,41.744,72.951,0.634,0.224,0.228,0.101,12.606
std,2059.449,18023.843,394.384,21.583,19.578,0.347,0.184,0.120,0.100,5.557
min,5.000,6.000,0.000,3.221,27.298,0.041,0.000,0.000,0.000,0.000
25%,10.000,43.000,1.362,25.409,55.987,0.396,0.051,0.151,0.022,9.000
50%,29.000,149.000,12.024,35.735,69.014,0.643,0.219,0.217,0.084,13.000
75%,150.500,986.000,29.557,52.622,94.729,0.835,0.352,0.288,0.144,17.000
max,21566.000,254119.000,8933.571,94.856,99.810,2.812,0.898,0.744,0.662,23.000


In [6]:
# 5) Trip-level features
trips = F.trip_features(prepared)
print(f"trips: {len(trips):,}")
trips[["duration_s","distance_km","dwell_s","speed_mean","speed_p95","stopgo_cv",
       "n_dwell_episodes"]].describe().round(3)

trips: 58,686


,duration_s,distance_km,dwell_s,speed_mean,speed_p95,stopgo_cv,n_dwell_episodes
count,58686.000,58671.000,58671.000,58671.000,58671.000,58406.000,58671.000
mean,728.234,4.895,190.889,25.611,55.918,0.791,4.528
std,649.658,3.933,390.404,11.140,19.826,0.226,3.359
min,0.000,0.001,0.000,0.181,0.181,0.000,0.000
25%,420.000,2.369,60.000,17.770,42.015,0.653,2.000
50%,615.000,3.873,120.000,23.560,51.469,0.763,4.000
75%,870.000,6.275,210.000,31.714,68.002,0.890,6.000
max,32460.000,116.418,28245.000,99.510,99.943,4.920,72.000


In [7]:
# 6) Weekly activity profile (cells x 168), row-normalised for shape clustering in Stage 5.
# Restricted to the cells we keep, so the profile and the feature table describe the same places.
mask = prepared["h3_cell"].isin(kept["h3_cell"])
profile = F.hour_of_week_profile(prepared[mask])
profile_counts = F.hour_of_week_profile(prepared[mask], normalise=False)
print("profile shape:", profile.shape, "| row sums:", round(float(profile.sum(axis=1).mean()), 4))


profile shape: (602, 168) | row sums: 1.0


In [8]:
# 7) Standardisation. Counts are log1p-transformed first because they are heavy-tailed.
FEATURE_COLS = ["n_visits","n_points","n_vehicles","dwell_s_per_visit","speed_mean","speed_p95",
                "speed_cv","slow_share","night_share","am_peak_share","pm_peak_share",
                "weekend_share","hour_sin_mean","hour_cos_mean"]
LOG_COLS = ("n_visits","n_points","n_vehicles","dwell_s_per_visit")

scaled, zparams = F.standardise(kept, FEATURE_COLS, log1p_cols=LOG_COLS)

# What scaling actually changed: raw features live on wildly different scales.
raw_sd = kept[FEATURE_COLS].std(ddof=0)
z_sd = scaled[["z_" + c for c in FEATURE_COLS]].std(ddof=0)
pd.DataFrame({"raw_sd": raw_sd.round(3).values,
              "scaled_sd": z_sd.round(3).values}, index=FEATURE_COLS)

,raw_sd,scaled_sd
n_visits,2057.738,1.0
n_points,18008.867,1.0
n_vehicles,136.629,1.0
dwell_s_per_visit,394.057,1.0
speed_mean,21.565,1.0
speed_p95,19.562,1.0
speed_cv,0.347,1.0
slow_share,0.184,1.0
night_share,0.120,1.0
am_peak_share,0.100,1.0


In [9]:
# 8) Sanity checks that must pass before trusting any of this
checks = {
    "haversine vs known distance": F.check_haversine_against_reference().to_dict("records"),
    "n_cells_used": len(kept),
    "n_trips": int(len(trips)),
    "step_valid_share": round(float(prepared["step_valid"].mean()), 4),
    # hour_sin/hour_cos must be almost uncorrelated across cells, otherwise the
    # encoding is not doing its job (they are orthogonal on a full circle).
    "corr_hour_sin_cos": round(float(kept["hour_sin_mean"].corr(kept["hour_cos_mean"])), 4),
    "peak_hour_range": [int(kept["peak_hour"].min()), int(kept["peak_hour"].max())],
    "cells_with_nan_features": int(scaled[["z_" + c for c in FEATURE_COLS]].isna().any(axis=1).sum()),
    "zero_variance_features": [c for c in FEATURE_COLS if zparams[c].get("sd", 1) == 0],
}
print(json.dumps(checks, indent=1))

{
 "haversine vs known distance": [
  {
   "route": "Lisbon->Porto",
   "haversine_km": 274.3,
   "reference_km_approx": 274.0
  }
 ],
 "n_cells_used": 602,
 "n_trips": 58686,
 "step_valid_share": 0.9566,
 "corr_hour_sin_cos": -0.0613,
 "peak_hour_range": [
  0,
  23
 ],
 "cells_with_nan_features": 0,
 "zero_variance_features": []
}


In [10]:
# 9) Figure: where activity happens (cell centroids, size ~ visits)
fig, ax = plt.subplots(figsize=(7, 6))
sizes = 4 + 40 * (kept["n_visits"] / kept["n_visits"].max())
ax.scatter(kept["lon_centroid"], kept["lat_centroid"], s=sizes, c="#111111", alpha=0.65, linewidths=0)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(f"Porto - H3 res 8 cells with >= {MIN_VISITS} trips ({len(kept)} cells)\nmarker size ~ trip visits")
ax.set_aspect("equal", adjustable="datalim")
fig.tight_layout(); fig.savefig(FIG / "stage3_cell_activity.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage3_cell_activity.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage3_cell_activity.png


In [11]:
# 10) Figure: activity by hour, two honest summaries.
# (a) mean of per-cell normalised profiles - shows the *shape* but is pulled by a few
#     nocturnal cells; (b) share of cells with any activity - shows how widespread it is.
mat = profile.to_numpy()
hour_idx = profile.columns.to_numpy() % 24
day_idx = profile.columns.to_numpy() // 24
cnt = profile_counts.to_numpy()

wk = np.array([mat[:, (day_idx < 5) & (hour_idx == h)].mean() for h in range(24)])
we = np.array([mat[:, (day_idx >= 5) & (hour_idx == h)].mean() for h in range(24)])
wk_active = np.array([(cnt[:, (day_idx < 5) & (hour_idx == h)] > 0).mean() for h in range(24)])
we_active = np.array([(cnt[:, (day_idx >= 5) & (hour_idx == h)] > 0).mean() for h in range(24)])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(range(24), wk, color="#111111", lw=2, label="weekday (Mon-Fri)")
axes[0].plot(range(24), we, color="#9A9A9A", lw=2, label="weekend")
axes[0].set_title("mean share of weekly activity per cell")
axes[0].set_ylabel("share of that cell's weekly activity")
axes[1].plot(range(24), wk_active, color="#111111", lw=2, label="weekday")
axes[1].plot(range(24), we_active, color="#9A9A9A", lw=2, label="weekend")
axes[1].set_title("share of cells with any activity")
axes[1].set_ylabel("fraction of cells active")
for ax in axes:
    ax.set_xticks(range(0, 24, 3)); ax.set_xlabel("hour of day (Europe/Lisbon)")
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "stage3_weekly_profile.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage3_weekly_profile.png")
print("mean-profile peak hour  - weekday:", int(wk.argmax()), "weekend:", int(we.argmax()))
print("cell-reach peak hour   - weekday:", int(wk_active.argmax()), "weekend:", int(we_active.argmax()))
print("cells active at 02:00  - weekday:", round(float(wk_active[2]),3), "weekend:", round(float(we_active[2]),3))


saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage3_weekly_profile.png
mean-profile peak hour  - weekday: 17 weekend: 18
cell-reach peak hour   - weekday: 11 weekend: 5
cells active at 02:00  - weekday: 0.25 weekend: 0.371


In [12]:
# 11) Figure: the cyclic encoding, drawn as the circle it is
hours = np.arange(24)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.scatter(np.cos(2*np.pi*hours/24), np.sin(2*np.pi*hours/24), c="#111111", s=28)
for h in hours:
    ax.annotate(f"{h:02d}", (np.cos(2*np.pi*h/24), np.sin(2*np.pi*h/24)),
                textcoords="offset points", xytext=(5, 4), fontsize=8, color="#555555")
ax.axhline(0, color="#D9D9D9", lw=1); ax.axvline(0, color="#D9D9D9", lw=1)
ax.set_aspect("equal"); ax.set_title("hour -> (cos, sin): 23:00 and 00:00 are neighbours")
ax.set_xlabel("cos(2*pi*h/24)"); ax.set_ylabel("sin(2*pi*h/24)")
fig.tight_layout(); fig.savefig(FIG / "stage3_cyclic_encoding.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage3_cyclic_encoding.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage3_cyclic_encoding.png


In [13]:
# 12) Figure: why scaling matters - raw spreads differ by orders of magnitude
fig, ax = plt.subplots(figsize=(8, 3.2))
x = np.arange(len(FEATURE_COLS))
ax.bar(x - 0.2, raw_sd.values, width=0.4, color="#111111", label="raw sd")
ax.bar(x + 0.2, z_sd.values, width=0.4, color="#D9D9D9", label="after z-score")
ax.set_xticks(x); ax.set_xticklabels(FEATURE_COLS, rotation=60, ha="right", fontsize=8)
ax.set_ylabel("standard deviation"); ax.set_title("same features, before and after standardisation")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "stage3_scaling_effect.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage3_scaling_effect.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage3_scaling_effect.png


In [14]:
# 13) Persist intermediates. Everything carries data_source so later stages cannot lose it.
prepared.to_parquet(PROC / "porto_points_prepared.parquet", index=False)
kept.to_parquet(PROC / "cells_features.parquet", index=False)
scaled.to_parquet(PROC / "cells_features_scaled.parquet", index=False)
trips.reset_index().to_parquet(PROC / "trip_features.parquet", index=False)
profile.to_parquet(PROC / "hour_of_week_profile.parquet")
profile_counts.to_parquet(PROC / "hour_of_week_counts.parquet")

meta_out = {
    "stage": 3,
    "data_source": "porto_taxi_uci339",
    "synthetic": False,
    "limitation": meta["limitation"],
    "source_meta": meta,
    "cleaning": report,
    "bbox": bbox,
    "min_visits_filter": MIN_VISITS,
    "n_cells_total": int(len(cells)),
    "n_cells_used": int(len(kept)),
    "n_trips": int(len(trips)),
    "feature_cols": FEATURE_COLS,
    "log1p_cols": list(LOG_COLS),
    "z_params": zparams,
    "checks": checks,
}
(PROC / "features_meta.json").write_text(json.dumps(meta_out, indent=2, default=str))
print("wrote", PROC / "features_meta.json")

wrote /home/ashira/Documents/Projects/Fleetsense/data/processed/features_meta.json


## Reading the result

* The weekly profile is the first **real** signal in this project: if the weekday curve peaks around 08-09 and
  18-19 local time while the weekend curve is flatter and later, the taxi demand behaves like a city, which is
  the check that the timezone conversion and the sampling are not broken.
* `step_valid` is the share of pings whose step can be trusted; it excludes the few steps flagged as physically
  impossible in Stage 2, so speed features cannot be dragged by a single teleport.
* `corr_hour_sin_cos` should be close to zero: on a full circle the two components are orthogonal. If it were
  strongly non-zero, our encoded features would be redundant and clustering would over-weight time of day.
* Cells with fewer than 5 trips are excluded from clustering. Their profiles would be one vehicle's noise, and
  including them would create a "rare cell" cluster that means nothing.